<a href="https://colab.research.google.com/github/Emmanuel-Yerbo/GIS/blob/main/Python%20for%20Urban%20Analysis/Chapter-3/ch3_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Buildings on OSM: Categorical and Continuous Maps

Learning Objectives
- Extract building footprints from OSM
- Classify buildings by type(residential, commercial)
- Map building as a continues variable (choropleth)
- Explore height/floor data (where available)
- Produce the hero visualization

In [ ]:
!pip install -q osmnx geopandas contextily mapclassify

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import contextily as cx
import mapclassify
from pathlib import Path

In [ ]:
# STUDY CONFIGURATION
study_area = "Accra, Ghana"
crs_mercator = "EPSG:3857"
crs_utm = "EPSG:32630"
dark_bg = "#0e1117"
accent_cmap = "YlOrRd"
building_tags = {"building":True}



In [ ]:
# FOLDERS
data_dir = Path('data')
figures_dir = Path("outputs") / "fgures"
for d in [data_dir, figures_dir]:
  d.mkdir(parents =True, exist_ok=True)

## Extract Building Footprints

Osm may return point nodes or linestrings alongside polygons. We filter to keep only polygons and Multipolygon

In [ ]:
print('Pulling Building footprints (this may take some time')
buildings = ox.features_from_place(study_area, tags = building_tags)
print(f"Raw features returned: {len(buildings)}")

In [ ]:
# Keeping only polygons
buildings_poly = buildings[
    buildings.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

print(f"Polygon footprints: {len(buildings_poly)}")

### Computing Footprint Area
We project to UTM ZONE 30 N -   a meter based CRS that covers Accra- before computing area. Never compute area in LAT/LON

In [ ]:
building_proj = buildings_poly.to_crs(crs_utm)
buildings_poly["area_m2"] = building_proj.geometry.area.round(1)

print("Footprint Area Statistics")
print(buildings_poly['area_m2'].describe().to_string())

## Classify Buildings by Type
[yes, residential, commercial, industrial, retail, school, church]

In [ ]:
if 'building' in buildings_poly.columns:
    buildings['bldg_type'] = buildings['building'].fillna('unknown')
    print('── Building type breakdown ──')
    type_counts = buildings['bldg_type'].value_counts().head(15)
    for btype, n in type_counts.items():
        print(f'   {btype:25s}  {n:>6}')

### Explore Height and Floors Data

reality checks for africa, building height and floor cont (building:levels) tags are very sparse in most african cites. this is normal not a limitation. Footprint area is the most universally available building metric

In [ ]:
for col in ['height', 'building:levels']:
  if col in buildings.columns:
    available = buildings[col].notna().sum()
    pct = 100 * available / len(buildings)
    print(f"{col}: {available} records ({pct:.1f}%)")
  else:
    print(f"{col}:not present in data")

# Convert to numeric where posible
if 'building:levels' in buildings.columns:
  buildings['levels'] = pd.to_numeric(buildings['building:levels'], errors='coerce')
else:
  buildings['levels'] = np.nan
if 'height' in buildings.columns:
  buildings['height_m'] = pd.to_numeric(buildings['height'], errors='coerce')
else:
  buildings['height_m'] = np.nan

In [ ]:
# Save Processed Buildings
save_cols = [c for c in ['geometry', 'area_m2', 'bldg_type', 'levels', 'height_m'] if c in buildings.columns]
buildings_save = buildings[save_cols].copy()
buildings_path = data_dir / 'buildings.gpkg'
buildings_save.to_file(buildings_path, driver='GPKG')
print(f'💾 Buildings saved → {buildings_path}')

In [ ]:
# HERO VISUALIZATION
print("creating building footprint map(area choropleth)......")
boundary = ox.geocode_to_gdf(study_area)

# Filtering out extremely large polygons(likely mapping artifacts)
q99 = buildings_poly['area_m2'].quantile(0.99)
bldg_plot = buildings_poly[buildings_poly['area_m2']<=q99].copy()

In [ ]:
fig, ax = plt.subplots(1,1, figsize = (16,16))
fig.patch.set_facecolor(dark_bg)
ax.set_facecolor(dark_bg)

# Plotting Buildings Footprints
bldg_plot.to_crs(crs_mercator).plot(
    ax = ax,
    column = 'area_m2',
    cmap = accent_cmap,
    scheme = 'quantiles',
    k =7,
    linewidth = 0.1,
    edgecolor = '#333333',
    alpha = 0.9,
    legend = False
)

# boundary
boundary.to_crs(crs_mercator).plot(
    ax=ax,
    facecolor = 'none',
    edgecolor = 'white',
    linewidth=2.5,
    zorder = 10
)

# dark basemap
cx.add_basemap(ax, source = cx.providers.CartoDB.DarkMatter, zoom = 14)

ax.set_axis_off()
ax.set_title(
    f"Building Footprint by Area - {study_area}",
    fontsize = 18, fontweight = 'bold', color = 'white', pad = 20
)

# Custom color bar
sm  = plt.cm.ScalarMappable(
    cmap = accent_cmap,
    norm = mcolors.Normalize(vmin=bldg_plot['area_m2'].min(),
                             vmax=bldg_plot['area_m2'].max())
)

sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, shrink = 0.6)
cbar.set_label('Footprint Area (m2)', color= 'white', fontsize=11)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color = 'white')

fig.tight_layout()
fig.savefig(figures_dir / 'ch3_building_area.png', dpi=150,
         bbox_inches = 'tight', facecolor=dark_bg)
plt.show()


In [ ]:
from matplotlib.figure import FigureBase
# Categorical Mapping
type_palette = {
    'yes':           '#ffc107',
    'residential':   '#4caf50',
    'commercial':    '#2196f3',
    'industrial':    '#ff5722',
    'retail':        '#e91e63',
    'church':        '#9c27b0',
    'school':        '#00bcd4',
    'mosque':        '#8bc34a',
    'hospital':      '#f44336',
    'warehouse':     '#795548',
}


In [ ]:
print(f"Columns in bldg_plot: {bldg_plot.columns.tolist()}")
print(f"Does 'bldg_type' exist? {'bldg_type' in bldg_plot.columns}")

# If missing, we can re-assign it from the source if indices match
if 'bldg_type' not in bldg_plot.columns:
    # Joining the classification back if necessary
    bldg_plot = bldg_plot.join(buildings[['bldg_type']], how='left')
    print("Re-joined 'bldg_type' column. Now re-run the previous cell.")

In [ ]:
if 'bldg_type' in bldg_plot.columns:
    bldg_plot['color'] = bldg_plot['bldg_type'].map(type_palette).fillna('#757575')

    fig, ax = plt.subplots(1, 1, figsize=(16, 16))
    fig.patch.set_facecolor(dark_bg)
    ax.set_facecolor(dark_bg)

    # Plotting
    bldg_plot.to_crs(crs_mercator).plot(
        ax=ax, color=bldg_plot['color'],
        linewidth=0.1, edgecolor='#222222', alpha=0.85,
    )

    boundary.to_crs(crs_mercator).plot(
        ax=ax, facecolor='none', edgecolor='white', linewidth=2.5, zorder=10
    )

    cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=14)
    ax.set_axis_off()
    ax.set_title(
        f'Building Types — {study_area}',
        fontsize=18, fontweight='bold', color='white', pad=20,
    )

    handles = [mpatches.Patch(color=c, label=t) for t, c in type_palette.items()]
    ax.legend(
        handles=handles, loc='lower right', fontsize=8,
        facecolor='#1a1a2e', edgecolor='white',
        labelcolor='white', framealpha=0.9, ncol=2
    )

    fig.tight_layout()
    plt.show()
else:
    print("Error: bldg_type still not found in bldg_plot.")